# Data control panel

Local control panel for checking data freshness, updating the tournament master file, and running the project smoke check.

In [ ]:
from __future__ import annotations

import subprocess
import sys
import threading
from pathlib import Path

from IPython.display import Markdown, display
import ipywidgets as widgets


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Could not find project root.')


PROJECT_ROOT = find_project_root()
PYTHON = sys.executable
BUTTONS: list[widgets.Button] = []


def set_buttons_disabled(disabled: bool) -> None:
    for button in BUTTONS:
        button.disabled = disabled


def append_log(text: str) -> None:
    with output:
        print(text, end='' if text.endswith('\n') else '\n')


def run_project_command_streaming(title: str, args: list[str]) -> None:
    set_buttons_disabled(True)
    progress.value = 0
    progress.bar_style = 'info'
    status_html.value = f'<b>{title}</b>: running...'

    with output:
        output.clear_output()
        display(Markdown(f'### {title}'))
        print(f'$ {PYTHON} ' + ' '.join(args))
        print('')

    try:
        process = subprocess.Popen(
            [PYTHON, *args],
            cwd=PROJECT_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )

        line_count = 0
        assert process.stdout is not None
        for line in process.stdout:
            line_count += 1
            progress.value = min(99, 5 + line_count)
            append_log(line)

        return_code = process.wait()
        if return_code == 0:
            progress.value = 100
            progress.bar_style = 'success'
            status_html.value = f'<b>{title}</b>: done'
        else:
            progress.value = 100
            progress.bar_style = 'danger'
            status_html.value = f'<b>{title}</b>: failed with code {return_code}'
    except Exception as exc:
        progress.value = 100
        progress.bar_style = 'danger'
        status_html.value = f'<b>{title}</b>: failed'
        append_log(str(exc))
    finally:
        set_buttons_disabled(False)


def show_command(title: str, args: list[str]) -> None:
    thread = threading.Thread(target=run_project_command_streaming, args=(title, args), daemon=True)
    thread.start()


status_button = widgets.Button(description='Refresh status', button_style='info')
tournaments_button = widgets.Button(description='Update tournaments master', button_style='warning')
calendar_button = widgets.Button(description='Update RTT calendar', button_style='warning')
verify_button = widgets.Button(description='Verify project', button_style='success')
pipeline_button = widgets.Button(description='Run full pipeline', button_style='danger')
BUTTONS.extend([status_button, tournaments_button, calendar_button, pipeline_button, verify_button])

progress = widgets.IntProgress(value=0, min=0, max=100, description='Progress:', bar_style='')
status_html = widgets.HTML(value='Ready')
output = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '8px'})

status_button.on_click(lambda _: show_command('Data status', ['scripts/data_status.py', '--write-manifest']))
tournaments_button.on_click(lambda _: show_command('Tournament master update', ['scripts/bootstrap_tournaments_master.py']))
calendar_button.on_click(lambda _: show_command('RTT calendar update', ['scripts/parse_rtt_calendar.py']))
pipeline_button.on_click(lambda _: show_command('Full update pipeline', ['scripts/run_full_pipeline.py']))
verify_button.on_click(lambda _: show_command('Project verification', ['scripts/verify_project.py']))

display(Markdown(f'Project root: `{PROJECT_ROOT}`'))
display(widgets.HBox([status_button, tournaments_button, calendar_button, pipeline_button, verify_button]))
display(widgets.VBox([status_html, progress]))
display(output)
show_command('Data status', ['scripts/data_status.py', '--write-manifest'])
